In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense
import statistics
import collections
import time
from tqdm import trange
import matplotlib.pyplot as plt

import gym

In [2]:
env = gym.make("MountainCar-v0")

# Set seed for experiment reproducibility
seed = 42
env.reset(seed=seed)
tf.random.set_seed(seed)
np.random.seed(seed)

# Valor épsilon pequeno para estabilizar operações de divisão
eps = np.finfo(np.float32).eps.item()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


In [16]:
class Model:
  def __init__(self, state_size, action_size):
      self.state_size = state_size
      self.action_size = action_size
      self.params = [np.random.rand(state_size, action_size), np.random.rand(state_size, 1)]

  def __call__(self, state):
      action_logits = state @ self.params[0]
      value = state @ self.params[1]
      return action_logits, value

In [4]:
def np_env_step(action):
    state, reward, done, _ = env.step(action)
    return np.array(state), np.array(reward), np.array(done)

In [31]:
def run_episode(initial_state, model, max_steps):

  action_probs =[]
  values = []
  rewards = []

  state = initial_state
  initial_state_shape = initial_state.shape

  for t in range(max_steps):
    # Converte o estado atual em um tensor de tamanho de lote 1 (necessário pelo modelo)
    state = np.expand_dims(state, 0)

    # Executa o modelo e obtém os logits de ação e o valor crítico
    action_logits_t, value = model(state)

    # Amostra a próxima ação a partir da distribuição de probabilidade de ação
    action_probs_t = np.exp(action_logits_t) / np.sum(np.exp(action_logits_t))
    action = np.random.choice(len(action_probs_t[0]), p=action_probs_t[0])

    # Armazena o valor crítico atual
    values.append(value[0, 0])

    # Armazena a probabilidade da ação escolhida
    action_probs.append(action_probs_t[0, action])

    # Aplica a ação ao ambiente para obter o próximo estado e recompensa
    state, reward, done = np_env_step(action)
    state = state.reshape(initial_state_shape)

    # Armazena a recompensa atual
    rewards.append(reward)

    # Se o episódio estiver concluído, saia do loop
    if state[0] >= 0.5:
        done = True

    if done :
        break

    # Converte listas em arrays do NumPy
    action_probs = np.array(action_probs)
    values = np.array(values)
    rewards = np.array(rewards)

    return action_probs, values, rewards

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [32]:
def run_episode(initial_state, model, max_steps):
    action_probs = []
    values = []
    rewards = []

    state = initial_state

    for t in range(max_steps):
        state = np.expand_dims(state, axis=0)
        action_logits_t, value = model(state)
        action = np.argmax(action_logits_t)
        action_probs_t = np.exp(action_logits_t) / np.sum(np.exp(action_logits_t))

        values.append(value)
        action_probs.append(action_probs_t[0, action])

        state, reward, done = np_env_step(action)
        rewards.append(reward)

        if state[0] >= 0.5:
            done = True

        if done:
            break

    return np.array(action_probs), np.array(values), np.array(rewards), state[0]

In [33]:
def get_expected_return(rewards, gamma, standardize=True):
    n = rewards.shape[0]
    rewards = rewards[::-1].astype(np.float32)
    discounted_sum = 0.0
    returns = np.zeros_like(rewards)

    for i in range(n):
        reward = rewards[i]
        discounted_sum = reward + gamma * discounted_sum
        returns[i] = discounted_sum

    returns = returns[::-1]

    if standardize:
        mean = np.mean(returns)
        std = np.std(returns) + 1e-8
        returns = (returns - mean) / std

    return returns

In [34]:
huber_loss = tf.keras.losses.Huber(reduction=tf.keras.losses.Reduction.SUM)

def compute_loss(action_probs, values, returns, delta=1.0):
    advantage = returns - values
    action_log_probs = np.log(action_probs)
    actor_loss = -np.sum(action_log_probs * advantage)
    critic_loss = huber_loss(returns, values, delta)
    return actor_loss + critic_loss

In [35]:
def compute_gradients(model, loss_fn, action_probs, values, returns):
    grads = []
    for param in model.params:
        original_param = param.copy()
        grad = np.zeros_like(param)
        h = 1e-5
        for idx, _ in np.ndenumerate(param):
            param[idx] += h
            loss_plus_h = loss_fn(action_probs, values, returns)
            param[idx] -= 2 * h
            loss_minus_h = loss_fn(action_probs, values, returns)
            param[idx] = original_param[idx]
            grad[idx] = (loss_plus_h - loss_minus_h) / (2 * h)
        grads.append(grad)
    return grads

In [36]:
def train_step(initial_state, model, optimizer, gamma, max_steps_per_episode):
    action_probs, values, rewards, _ = run_episode(initial_state, model, max_steps_per_episode)
    returns = get_expected_return(rewards, gamma)
    loss_fn = lambda ap, v, r: compute_loss(ap, v, r)
    loss = compute_loss(action_probs, values, returns)
    grads = compute_gradients(model, loss_fn, action_probs, values, returns)
    optimizer.update(grads)
    episode_reward = np.sum(rewards)
    return episode_reward

In [37]:
class AdamOptimizer:
    def __init__(self, params, lr=0.01, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def update(self, grads):
        self.t += 1
        lr_t = self.lr * np.sqrt(1 - self.beta2 ** self.t) / (1 - self.beta1 ** self.t)
        for i, (param, grad) in enumerate(zip(self.params, grads)):
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * grad
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (grad ** 2)
            param -= lr_t * self.m[i] / (np.sqrt(self.v[i]) + self.epsilon)

In [38]:
state_size=env.observation_space.shape[0]
action_size=env.action_space.n

model = Model(state_size, action_size)
optimizer = AdamOptimizer(model.params, lr=0.01)

In [39]:
max_episodes = 1500
max_steps_per_episode = 300

running_reward = 0
gamma = 0.99
episodes_reward = collections.deque(maxlen=max_episodes)
reward_history = []

# Loop de treinamento
with trange(max_episodes) as t:
    for i in t:
        initial_state = env.reset()
        state = initial_state
        episode_reward = 0

        for step in range(max_steps_per_episode):
            state = np.expand_dims(state, axis=0)
            action_probs, value = model(state)
            action = np.argmax(action_probs)
            state, reward, done = np_env_step(action)
            episode_reward += reward

            if state[0] >= 0.5:
                done = True

            if done:
                break
        episodes_reward.append(episode_reward)
        reward_history.append(episode_reward)
        running_reward = statistics.mean(episodes_reward)

        t.set_description(f'Episode {i}')
        t.set_postfix(episode_reward=episode_reward, running_reward=running_reward)

print(f'\nResolvido no episódio {i + 1}: recompensa média: {running_reward:.2f}!')

Episode 1499: 100%|██████████| 1500/1500 [00:35<00:00, 42.42it/s, episode_reward=-200, running_reward=-200]


Resolvido no episódio 1500: recompensa média: -200.00!
